# Lab 02 · Publish a Foundry model and make your first call

Publish a Microsoft Foundry model deployment through your AI Gateway, create a runtime API key, and call the model
with the OpenAI Python SDK. This notebook is the Python track of the [lab guide](README.md): it deploys the same
[infra/foundry.bicep](infra/foundry.bicep) and [infra/publish-model.bicep](infra/publish-model.bicep) as the Bicep track, then runs
the same checks.

**Deploy Foundry → Publish the model with Bicep → Read the key → Call → Verify**

## Architecture at a glance

| Component | What it does |
| --- | --- |
| Foundry account and `gpt-6-luna` deployment | The backend that serves the model. [infra/foundry.bicep](infra/foundry.bicep) creates them, unless you bring your own. |
| Foundry User role | Lets the gateway's managed identity call the account, so no Foundry keys are involved. |
| Model provider `foundry-models` | Tells the gateway where the account is and to authenticate with its managed identity. |
| Backend model `gpt-6-luna` | Registers the deployment under the provider, with a limit of 10,000 tokens per minute for each caller. |
| Alias `gpt-6-luna` | The stable name clients send in the `model` field. |
| Runtime API key `app-first-call` | Authenticates your app to the gateway. Notebooks 04 and 06 reuse it. |
| [infra/publish-model.bicep](infra/publish-model.bicep) | Declares the role assignment, provider, backend model, alias, and key. |
| [../shared/utils.py](../shared/utils.py) | Helpers that every lab notebook shares: settings, your `az login` sign-in, Bicep deployments, keys, and retries. |

![Request lifecycle: the gateway validates the key, runs policies, swaps in its managed identity, and forwards to Foundry](../assets/diagrams/request-lifecycle.svg)

## Prerequisites

- An Azure subscription where you can create resources (Contributor on the resource group) and assign roles
  (Owner, User Access Administrator, or Role Based Access Control Administrator).
- The [Azure CLI](https://learn.microsoft.com/cli/azure/install-azure-cli), signed in with `az login`. The notebook
  deploys Bicep with `az deployment group create`, and the CLI installs Bicep the first time it needs it.
- Python 3.10 or later with Jupyter, for example VS Code with the Python and Jupyter extensions.
- [01-provision-gateway.ipynb](../01-provision-gateway/01-provision-gateway.ipynb) has run, so the gateway exists.
- Quota for `gpt-6-luna` (`GlobalStandard`) in the gateway's region. Model inference is billed per use.

Run the cells in order, or click **Run All**. Each step is idempotent: rerun any cell and it updates the same resources.

In [ ]:
%pip install -q --upgrade azure-identity requests openai

### 0. Initialize the lab

The defaults match notebook 01, so `utils.Lab` finds the gateway without edits, and stops with a clear message if it
isn't there. The notebook creates a Foundry account with a name derived from your subscription. To use an account
that already has a `gpt-6-luna` deployment, set `AIGW_FOUNDRY_ACCOUNT` and `AIGW_FOUNDRY_RESOURCE_GROUP` before
you start the kernel.

In [ ]:
import sys
sys.path.insert(0, "../shared")   # helpers that every lab notebook shares
import utils

lab = utils.Lab("02-first-model-call")
deployment    = "gpt-6-luna"   # Foundry deployment, backend model, and alias name
model_version = "2026-09-22"

print(f"🏭 Foundry:        {lab.foundry_account} ({lab.foundry_resource_group})")
print(f"🤖 Model:          {deployment} {model_version}")

### 1. Create the Foundry account and model deployment

The gateway publishes models that already run in Foundry. [infra/foundry.bicep](infra/foundry.bicep) declares an
Azure AI Services account and a `gpt-6-luna` `GlobalStandard` deployment. Local key authentication stays disabled:
the gateway authenticates with its managed identity. If you set `AIGW_FOUNDRY_ACCOUNT`, the cell checks your account
and its deployment instead of deploying.

In [ ]:
lab.register_providers("Microsoft.CognitiveServices")

if lab.own_foundry_account:
    lab.require(lab.foundry_id, "Check AIGW_FOUNDRY_ACCOUNT and AIGW_FOUNDRY_RESOURCE_GROUP.", utils.COGNITIVE_API)
    lab.require(f"{lab.foundry_id}/deployments/{deployment}", f"Deploy {deployment} in the account first.",
                utils.COGNITIVE_API)
    print(f"♻️ Using your Foundry account {lab.foundry_account}")
else:
    lab.create_resource_group(lab.foundry_resource_group)
    foundry = lab.deploy("infra/foundry.bicep", resource_group=lab.foundry_resource_group,
                         accountName=lab.foundry_account, location=lab.location,
                         deploymentName=deployment, modelVersion=model_version)
    print(f"🏭 Foundry endpoint: {foundry['endpoint']}")

### 2. Publish the model through the gateway

[infra/publish-model.bicep](infra/publish-model.bicep) declares everything the gateway needs to serve the deployment:

- The **Foundry User** role for the gateway's managed identity on the account. The assignment name is derived from
  the account, gateway, and role, so a redeploy reuses it.
- A **model provider**, `foundry-models`, that tells the gateway where the account is and how to authenticate to it.
- A **backend model**, `gpt-6-luna`, that registers the deployment under the provider. It carries a token limit
  of 10,000 tokens per minute for each caller.
- An **alias**, `gpt-6-luna`: the name clients send in the request `model` field. It points at one or more
  backend models, so you can retarget it later without changing client code (lab 03).
- A **runtime API key**, `app-first-call`. Apps authenticate to the gateway with it, not with Foundry keys.

Role assignments can take a few minutes to apply. The first call in step 4 retries until they do.

In [ ]:
published = lab.deploy("infra/publish-model.bicep", gatewayName=lab.gateway_name, foundryAccountId=lab.foundry_id,
                       deploymentName=deployment, modelVersion=model_version)
for name, value in published.items():
    print(f"📤 {name:<30} {value}")

### 3. Read the runtime API key

The template creates the key but doesn't output its value, because deployment outputs aren't secret. `lab.api_key`
reads the primary value with `listSecrets`. Notebooks 04 and 06 reuse the same key. The value stays in memory: the
notebook never prints it in full or writes it to a file.

In [ ]:
api_key, _ = lab.api_key(published["apiKeyName"])
print(f"🔑 Key {published['apiKeyName']}: {api_key[:4]}…")

### 4. Call the model with the OpenAI SDK

Point the OpenAI client at the template's `modelEndpoint` output, `<gateway-url>/default/models/openai/v1`, and
pass the runtime key as `api_key`. The SDK sends it as `Authorization: Bearer <key>`, which the gateway accepts.
Don't send both `api-key` and `Authorization`: the gateway returns `400`. The `model` field holds the alias.

A brand-new key, model, or role assignment can take a few minutes to apply, so `utils.retry` retries `401`, `403`,
`404`, `429`, and `5xx` responses for up to 10 minutes.

In [ ]:
from openai import OpenAI

client = OpenAI(base_url=published["modelEndpoint"], api_key=api_key)
alias  = published["aliasName"]

response = utils.retry(lambda: client.responses.create(model=alias, input="Write a one-sentence status update."))
print(f"📨 Responses API:        {response.output_text}")

chat = utils.retry(lambda: client.chat.completions.create(
    model=alias, messages=[{"role": "user", "content": "Describe an AI gateway in one sentence."}]))
print(f"📨 Chat Completions API: {chat.choices[0].message.content}")
print(f"🔢 Tokens: {chat.usage.prompt_tokens} in, {chat.usage.completion_tokens} out")
assert response.output_text and chat.choices[0].message.content

### 5. Verify

Run the same checks as the lab guide:

- The model appears in the workspace's model list.
- A request with the key in the `api-key` header returns `200`.
- The same request without a key returns `401`. The gateway rejects it before any policy runs.
- A request for a name that isn't an alias returns `404` with `model_not_found`. Backend model and deployment names
  aren't callable unless an alias with that name exists.

In [ ]:
import requests

models = lab.arm("GET", f"{lab.workspace_id}/models")["value"]
for m in models:
    print(f"🤖 {m['name']:<20} {m['properties'].get('providerKind')}")
assert any(m["name"] == deployment for m in models)

chat_url = f"{published['modelEndpoint']}/chat/completions"
payload  = {"model": alias, "messages": [{"role": "user", "content": "ping"}]}

with_key = requests.post(chat_url, json=payload, headers={"api-key": api_key}, timeout=60)
print(f"📨 With key:      HTTP {with_key.status_code}")
assert with_key.status_code == 200, with_key.text[:500]

without_key = requests.post(chat_url, json=payload, timeout=60)
print(f"📨 Without key:   HTTP {without_key.status_code}")
assert without_key.status_code == 401

wrong_alias = requests.post(chat_url, json={**payload, "model": alias[:-1] + "x"}, headers={"api-key": api_key}, timeout=60)
print(f"📨 Wrong alias:   HTTP {wrong_alias.status_code} {wrong_alias.text[:120]}")
assert wrong_alias.status_code == 404 and "model_not_found" in wrong_alias.text
print("✅ All checks passed")

### 6. Clean up resources

Keep the gateway, model, alias, and key: notebooks 04, 06, and 11 reuse them. When you finish the series, uncomment
the lines below. Deleting the Foundry account soft-deletes it, so `lab.delete_foundry_account` also purges it, which
frees the name and stops any charges. It never deletes an account that you set with `AIGW_FOUNDRY_ACCOUNT`.

In [ ]:
provider_path = f"{lab.workspace_id}/modelProviders/{published['providerName']}"
role_path     = (f"{lab.foundry_id}/providers/Microsoft.Authorization/roleAssignments/"
                 f"{published['foundryUserRoleAssignmentName']}")

# Notebooks 04, 06, and 11 reuse these. Uncomment to remove them when you're done with the series.
# lab.delete(f"{lab.gateway_id}/apiKeys/{published['apiKeyName']}")
# lab.delete(f"{lab.workspace_id}/aliases/{alias}")
# lab.delete(f"{provider_path}/models/{published['modelName']}")
# lab.delete(provider_path)
# lab.delete(role_path, api_version=utils.ROLES_API)
# lab.delete_foundry_account()
if lab.arm("GET", provider_path):
    print(f"ℹ️ Kept {alias}, {published['providerName']}, key {published['apiKeyName']}, "
          f"and {lab.foundry_account} for the next notebooks")

## Troubleshooting

- **`401` on every call**: the key is new or wrong. Rerun step 3; new keys can take a minute to work. Send the key in
  `api-key` *or* `Authorization: Bearer`, not both (`400`).
- **`403` or `500` from the backend for several minutes**: the Foundry User assignment hasn't applied yet. Check it with
  `az role assignment list --scope <foundry-account-id> -o table`, then wait.
- **`RoleAssignmentExists` in step 2**: the gateway's identity already has Foundry User on the account under another
  assignment name, for example one you created in the portal. Remove it with
  `az role assignment delete --assignee <principal-id> --role 53ca6127-db72-4b80-b1b0-d745d6d5456d --scope <foundry-account-id>`,
  then rerun step 2.
- **`404` with `model_not_found`**: `model` must match an alias exactly. List aliases with
  `lab.arm("GET", f"{lab.workspace_id}/aliases")`.
- **`InsufficientQuota` in step 1**: set `AIGW_LOCATION` to the other preview region, pass a lower `capacity` to
  `lab.deploy`, or reuse an existing account with `AIGW_FOUNDRY_ACCOUNT`.
- **`CustomDomainInUse` or `FlagMustBeSetForRestore`**: a deleted account with the same name is in the soft-delete
  state. Purge it in the Azure portal (**Foundry** › **Manage deleted resources**), then rerun step 1.
- **`429`**: the deployment's tokens-per-minute quota is used up. Wait a minute, or redeploy with a higher `capacity`.

## Next steps

- Continue with [Federate MCP tools behind one endpoint](../04-federate-mcp-tools/README.md) and its notebook [04-federate-mcp-tools.ipynb](../04-federate-mcp-tools/04-federate-mcp-tools.ipynb).
- Throttle callers with [token limits](../06-token-limits/README.md) and its notebook [06-token-limits.ipynb](../06-token-limits/06-token-limits.ipynb).
- Add other providers behind stable names in [lab 03](../03-multi-provider-aliases/README.md).
- Docs: [Access models through AI Gateway](https://ai.gateway.azure.com/docs/model-access).
- Read this lab on the labs site: [https://azure.github.io/ai-gateway/labs/02-first-model-call/](https://azure.github.io/ai-gateway/labs/02-first-model-call/).